In [ ]:
import torch
from torch import nn
from torch import optim
from torch.nn.utils import parameters_to_vector
from torch.nn.functional import gelu
from torchvision import datasets, transforms

import torchattacks

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from copy import deepcopy
from itertools import permutations

from helpers import *

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

import matplotlib.animation as animation
from matplotlib.colors import ListedColormap

from collections import deque


In [ ]:
%config InlineBackend.figure_format = 'png'
device = "mps"

# Use this for Windows
device = "cuda" if torch.cuda.is_available() else "cpu"

# Functions and Classes

In [ ]:
# this class is used for the train_model_generator to use both yield and returns efficiently
# yield is for animations return is for everything else
class GeneratorWithReturn:
    """Wraps a generator so you can iterate it normally, and afterwards
    access whatever it `return`-ed via .value"""
    def __init__(self, gen):
        self.gen = gen
        self.value = None

    def __iter__(self):
        self.value = yield from self.gen

In [ ]:
def get_grid(input_dim, num_points_per_dim=100):

    #Classify on 100^n grid, contrained to a input_dim hypersphere
    n_arrays = [np.linspace(-1, 1, num_points_per_dim) for _ in range(input_dim)]
    grid = np.meshgrid(*n_arrays, indexing='ij')

    # keep points within the unit hypersphere
    within_radius = np.sum(np.array(grid)**2, axis=0) <= 1
    grid = [g[within_radius] for g in grid]
    # grid is now a list of input_dim arrays, each of shape (num_points,) inside the hypersphere

    # stack the arrays to create a 2D array of shape (num_points, input_dim)
    X = torch.Tensor(np.column_stack(grid)).to(device)
    return X

In [ ]:
def shannon_stability(cat_all, output_dim):
    # cat_all.shape = (len(X), output_dim)
    # get % of a point being classified as a class
    cat_all = cat_all / cat_all.sum(dim=1).unsqueeze(1)

    # stability of each point on grid from Shannon formula (len(X), 1)
    # elementwise mult -> cat_all * torch.log(cat_all + 1e-10) 
    stability = 1 + torch.sum(cat_all * torch.log(cat_all + 1e-10) / np.log(output_dim), dim=1)

    return stability.detach().cpu().numpy()

In [ ]:
# train model - stability calculated with only the last 100 trainings
def train_model(seed, hidden_dim, n_epochs, batch_size, amount_data, 
                input_dim=2, output_dim=3, num_points_per_dim=100, history_limit = 100):
    train_loader = hypersphere_data(input_dim, output_dim, amount_data, 
                                    batch_size = batch_size, seed = seed)    
    torch.manual_seed(seed)
    
    model = MLP((input_dim, *hidden_dim, output_dim), bias=True, activation = nn.GELU)
    model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-1)

    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)

    cat_last = None 
    perc_change = [] 
    predictions_history = deque(maxlen=history_limit)
    cat_all = torch.zeros(len(X), output_dim).to(device)
    loss_history = []

    for _ in range(n_epochs):
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)

            outputs = model(data)
            loss = criterion(outputs, target)
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_history.append(loss.item())

            with torch.no_grad():
                cat_next = model(X).argmax(dim=1)
                if cat_last != None:
                    perc_change.append( (cat_last != cat_next).sum().item() / len(X) )
                cat_last = cat_next

                # instead of using this
                # for i in range(output_dim):
                #     cat_all[cat_next == i, i] += 1
                
                # Append the current predictions to the history queue
                # (Moving to CPU prevents GPU memory leaks over time if X is huge)
                predictions_history.append(cat_next.cpu())  

                # initialize 'cat_all' to rebuild
                cat_all = torch.zeros((len(X), output_dim), dtype=torch.int32)
                
                # Stack the history along a new dimension to get shape (history_len, len(X))
                history_tensor = torch.stack(list(predictions_history))
                
                # Use scatter to count occurrences efficiently across the rolling window
                # scatter_add_ acts like a highly optimized batch version of the for-loop
                # add 1 at the index given by the history_tensor.T (add one history_len times at appropriate index)
                ones = torch.ones_like(history_tensor.transpose(0, 1), dtype=torch.int32)
                cat_all.scatter_add_(dim=1, index=history_tensor.transpose(0, 1), src=ones) # src = source

                # Calculate your stability based on the rolling 100-step history
                stab = shannon_stability(cat_all, output_dim)

                # Get weights and biases
                weights_bias = []
                for i in range(1, len(model.net), 2):
                    weights_bias.append([model.net[i].weight.detach().cpu(), model.net[i].bias.detach().cpu()])

                yield stab, cat_next, weights_bias

    return model, perc_change, stab, loss_history

In [ ]:
def animate_weights(gen, figsize_per_layer=(3, 3), cmap="RdBu_r"):
    wrapped = GeneratorWithReturn(gen)

    def combine(weight, bias):
        w_np = weight.numpy() if hasattr(weight, "numpy") else np.asarray(weight)
        b_np = bias.numpy() if hasattr(bias, "numpy") else np.asarray(bias)
        return np.concatenate([w_np, b_np[:, None]], axis=1)

    # Pull ALL frames in a single continuous iteration -- no separate next()+for
    all_combined = []
    for stab, cat_next, weights_bias in wrapped:
        combined = [combine(w, b) for w, b in weights_bias]
        all_combined.append(combined)

    final_value = wrapped.value  # populated now, since wrapped is fully exhausted
    n_layers = len(all_combined[0])

    fig, axes = plt.subplots(1, n_layers,
                              figsize=(figsize_per_layer[0]*n_layers, figsize_per_layer[1]))
    if n_layers == 1:
        axes = [axes]

    ims = []
    for i, (ax, mat) in enumerate(zip(axes, all_combined[0])):
        im = ax.imshow(mat, cmap=cmap, aspect="auto")
        ax.set_title(f"Layer {i+1}\n{mat.shape[0]}×{mat.shape[1]-1} + bias")
        ax.set_xticks([]); ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        ims.append(im)

    layer_mins = [min(frame[i].min() for frame in all_combined) for i in range(n_layers)]
    layer_maxs = [max(frame[i].max() for frame in all_combined) for i in range(n_layers)]
    for im, vmin, vmax in zip(ims, layer_mins, layer_maxs):
        im.set_clim(vmin, vmax)

    def update(frame_idx):
        for im, mat in zip(ims, all_combined[frame_idx]):
            im.set_data(mat)
        return ims

    ani = animation.FuncAnimation(
        fig, update, frames=len(all_combined), interval=100, blit=False
    )
    plt.tight_layout()
    plt.close(fig)
    return ani, final_value

In [ ]:
def animate_stability(gen,# seed, hidden_dim, n_epochs, batch_size, amount_data,
                      input_dim=2, output_dim=3, num_points_per_dim=100,
                      interval=50, overlay_seed=10):
    # interval = delay time in milisec for each frame
    # overlay_seed = seed for overlay scatter of another generated hypersphere data ()

    X = get_grid(input_dim, num_points_per_dim=num_points_per_dim)
    X_np = X.cpu().numpy() # turn grid into numpy array

    # Put generator from train_model_ in wrapper
    # gen = train_model_gen(seed, hidden_dim, n_epochs, batch_size, amount_data,
    #                    input_dim, output_dim, num_points_per_dim)
    wrapped = GeneratorWithReturn(gen)

    fig, (ax_bound, ax_stab) = plt.subplots(1, 2, figsize=(14, 6))

    # Get colors for the output classes
    class_cmap = ListedColormap(plt.cm.tab10.colors[:output_dim])

    # Initialize plots
    # Decision boundary scatter (colors updated each frame) 
    # use colormap instead of plotting 3 times...
    scat_bound = ax_bound.scatter(X_np[:, 0], X_np[:, 1], s=1, marker="s",
                                   c=np.zeros(len(X_np)), cmap=class_cmap,
                                   vmin=0, vmax=output_dim - 1)
    ax_bound.set_title("Decision boundary")
    ax_bound.set_xticks([]); ax_bound.set_yticks([])

    # Stability scatter 
    scat_stab = ax_stab.scatter(X_np[:, 0], X_np[:, 1], s=1, marker="s",
                                 c=np.zeros(len(X_np)), cmap="Spectral",
                                 vmin=0, vmax=1)
    ax_stab.set_title("Decision stability")
    ax_stab.figure.colorbar(scat_stab, ax=ax_stab)
    ax_stab.set_xticks([]); ax_stab.set_yticks([])

    # Create overlay scatter plot from data points generated by a different seed hypersphere_data
    # Plot shows the true labels for reference, nothing to do with MLP training stuff
    pt, lab = next(iter(hypersphere_data(input_dim, output_dim, 150,
                                          batch_size=150,
                                          seed=overlay_seed)))
    pt_np = pt.cpu().numpy() # turn into np
    lab_np = lab.cpu().numpy()

    # overlay the data in ax_bound, ax_stab for each output class
    for ax in (ax_bound, ax_stab):
        for i in range(output_dim):
            mask = lab_np == i
            ax.scatter(pt_np[mask, 0], pt_np[mask, 1], s=20, marker="s",
                       alpha=0.8, edgecolor="white", linewidth=0.7,
                       color=class_cmap(i))

    # function for updating the stab and cat_next each frame
    # which you can iterate using the generator from wrapped 
    # (produced by "yield" in the train_model_gen)
    def update(frame):
        stab, cat_next, _ = frame
        cat_np = cat_next.cpu().numpy()
        # if stab is a tensor, convert properly; otherwise just wrap it as an array
        stab_np = stab.cpu().numpy() if hasattr(stab, "cpu") else np.asarray(stab) # pretty sure it's a tensor bit just incase

        scat_bound.set_array(cat_np)
        scat_stab.set_array(stab_np)
        return scat_bound, scat_stab

    # Create animation object
    ani = animation.FuncAnimation(
        fig, update, frames=wrapped, interval=interval,
        blit=False, cache_frame_data=False
    )
    plt.close(fig)

    return ani, wrapped

In [ ]:
def plot_perc_change(perc_change, title="model0"):
    plt.figure(figsize = (8,2))
    plt.plot([100*x for x in perc_change])
    plt.xlabel("Training iteration")
    plt.ylabel("Decisions changed (%)")
    plt.title(title)
    plt.show()

# Small model

Only use small models because the animation requires saving all the frames from each training to find the min and max weight/bias to generate the appropriate heatmap - this uses a lot of memory.

In [ ]:
gen = train_model(seed=5, hidden_dim=(4,), input_dim=2, output_dim=3, n_epochs=30, amount_data=100, batch_size=20, num_points_per_dim=100)
ani, wrapped = animate_weights(gen)

ani.save('ani/ani_weights_2_4_3_seed5_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/ani_weights_2_4_3_seed5_cat100.mp4', embed=True)

In [ ]:
gen = train_model(seed=5, hidden_dim=(4,), input_dim=2, output_dim=3, n_epochs=30, amount_data=100, batch_size=20, num_points_per_dim=100)
ani, wrapped = animate_stability(gen)

ani.save('ani/ani_stability_2_4_3_seed5_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/ani_stability_2_4_3_seed5_cat100.mp4', embed=True)

In [ ]:
gen = train_model(seed=5, hidden_dim=(16,64,32), input_dim=2, output_dim=3, n_epochs=30, amount_data=100, batch_size=20, num_points_per_dim=100)
ani, wrapped = animate_weights(gen)

ani.save('ani/ani_weights_2_16_64_32_3_seed5_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/ani_weights_2_16_64_32_3_seed5_cat100.mp4', embed=True)

In [ ]:
gen = train_model(seed=5, hidden_dim=(16,32,64), input_dim=2, output_dim=3, n_epochs=60, amount_data=100, batch_size=20, num_points_per_dim=100)
ani, wrapped = animate_stability(gen)

ani.save('ani/ani_stability_2_16_64_32_3_seed5_cat100.mp4', writer='ffmpeg', fps=15)
from IPython.display import Video
Video('ani/ani_stability_2_16_64_32_3_seed5_cat100.mp4', embed=True)

In [ ]:
model, perc_change, stab, loss_history = wrapped.value
model

In [ ]:
# data, (#data, in_dim)
X = get_grid(input_dim=2)

# first layer, (#data,in_dim) @ (in_dim, 16) + (1, 16)       add bias to each #data
out1 = X @ model.net[1].weight.T + model.net[1].bias.unsqueeze(0) 
# GeLU 
out2 = model.net[2](out1)
# sec layer, (#data,16) @ (16, 64) + (1, 64)
out3 = out2 @ model.net[3].weight.T + model.net[3].bias.unsqueeze(0)
# GeLU 
out4 = model.net[4](out3)
# third layer, (#data,64) @ (64, 64) + (1, 64)
out5 = out4 @ model.net[5].weight.T + model.net[5].bias.unsqueeze(0)
# GeLU 
out6 = model.net[6](out5)
# output, (#data,64) @ (64, 3) + (1, 3)
out7 = out6 @ model.net[7].weight.T + model.net[7].bias.unsqueeze(0)

output0 = [X.detach().numpy(), out1.detach().numpy(), out2.detach().numpy(), out3.detach().numpy(),
           out4.detach().numpy(), out5.detach().numpy(), out6.detach().numpy(), out7.detach().numpy()]

Look at Norms - which is the analytical way of solving the boundaries

Primarily look at the penultimate-layer feature (second to last layer feature) and ouput layer

In [ ]:
norms = [np.linalg.norm(arr, axis=1) for arr in output0]

# Descriptive titles matching each extraction step
layer_names = [
    "Input X",
    "Layer 1 (Linear)",
    "Layer 1 (GeLU)",
    "Layer 2 (Linear)",
    "Layer 2 (GeLU)",
    "Layer 3 (Linear)",
    "Layer 3 (GeLU)",
    "Output (Linear)",
]

# Create 2x4 grid of subplots
fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(16, 7))
axes = axes.flatten()

# Plot bar charts
for i, norm_vec in enumerate(norms):
    ax = axes[i]
    sample_indices = np.arange(len(norm_vec))

    ax.bar(sample_indices, norm_vec, color="steelblue", alpha=0.85)

    title = layer_names[i] if i < len(layer_names) else f"Step {i}"
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.set_xlabel("Sample Index", fontsize=8)
    ax.set_ylabel("L2 Norm", fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Look at Max, which is how the MLP decides on the classes

In [ ]:
max_indices = output0[6].argmax(axis=1)

In [ ]:
sample_indices = np.arange(len(norm_vec))

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(sample_indices, output0[6].max(axis=1), color="steelblue", alpha=0.85)

ax.set_xlabel("Sample Indices")
ax.set_ylabel("Max Value")
ax.set_title("Maximum Values per Sample")
plt.tight_layout()

plt.show()

In [ ]:
output0[7].shape

In [ ]:
output0[7].argmax(axis=1).shape

In [ ]:
sample_indices = np.arange(len(norm_vec))

fig, ax = plt.subplots(figsize=(10, 5))

ax.bar(sample_indices, output0[7].argmax(axis=1), color="steelblue", alpha=0.85)

ax.set_xlabel("Sample Indices")
ax.set_ylabel("Max Value Indicie")
ax.set_title("Maximum Values per Sample")
plt.tight_layout()

plt.show()